Train meta learner bằng tập train, early stop bằng valid sau đó test trên tập test.

In [1]:
import pandas as pd
data_train = pd.read_csv(r"train.csv")
data_valid = pd.read_csv(r"val.csv")
data_test = pd.read_csv(r"test.csv")

In [2]:
import pandas as pd
import ast

def preprocess_dataframe(df):
    print("Bước 0: Tiền xử lý dữ liệu (Parsing & Sorting)...")
    
    # 1. Parse các cột List từ String sang Set (Tập hợp)
    list_columns = ['network_ips_src', 'network_ips_dst', 'network_ports_src', 'network_ports_dst']
    
    def safe_parse_to_set(val):
        try:
            if isinstance(val, str):
                parsed = ast.literal_eval(val)
                return set(parsed) if isinstance(parsed, list) else set()
            return set(val) if isinstance(val, list) else set()
        except (ValueError, SyntaxError):
            return set()

    for col in list_columns:
        print(f"  - Đang parse cột {col}...")
        df[f'{col}_set'] = df[col].apply(safe_parse_to_set)

    # 2. Chuyển đổi Timestamp chuẩn ISO 8601 sang số giây (Unix Epoch Time Float)
    print("  - Đang chuyển đổi Timestamp...")
    # format='ISO8601' giúp pandas phân tích chuỗi T...Z cực nhanh
    df['datetime_obj'] = pd.to_datetime(df['timestamp'], format='ISO8601', utc=True)
    
    # Lấy ra số giây dưới dạng float (chính xác đến microsecond)
    df['timestamp_num'] = df['datetime_obj'].astype('int64') / 1e9

    # 3. SẮP XẾP TĂNG DẦN THEO THỜI GIAN (Bắt buộc cho Sliding Window)
    df = df.sort_values(by='timestamp_num').reset_index(drop=True)
    
    return df

In [3]:
data_train = preprocess_dataframe(data_train)
data_valid = preprocess_dataframe(data_valid)
data_test = preprocess_dataframe(data_test)

Bước 0: Tiền xử lý dữ liệu (Parsing & Sorting)...
  - Đang parse cột network_ips_src...
  - Đang parse cột network_ips_dst...
  - Đang parse cột network_ports_src...
  - Đang parse cột network_ports_dst...
  - Đang chuyển đổi Timestamp...
Bước 0: Tiền xử lý dữ liệu (Parsing & Sorting)...
  - Đang parse cột network_ips_src...
  - Đang parse cột network_ips_dst...
  - Đang parse cột network_ports_src...
  - Đang parse cột network_ports_dst...
  - Đang chuyển đổi Timestamp...
Bước 0: Tiền xử lý dữ liệu (Parsing & Sorting)...
  - Đang parse cột network_ips_src...
  - Đang parse cột network_ips_dst...
  - Đang parse cột network_ports_src...
  - Đang parse cột network_ports_dst...
  - Đang chuyển đổi Timestamp...


In [4]:
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import GATv2Conv
from torch_geometric.utils import dropout_edge
import torch.nn as nn
import pandas as pd
import numpy as np
import gc

# mô hình GatEmbedder
class GAT_Embedder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, embedding_dim, num_classes, heads=4, edge_dropout=0.1, edge_dim=5):
        super(GAT_Embedder, self).__init__()
        self.edge_dropout = edge_dropout 
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=heads, dropout=0.1, edge_dim=edge_dim)
        self.bn1 = nn.BatchNorm1d(hidden_channels * heads)
        self.conv2 = GATv2Conv(hidden_channels * heads, embedding_dim, heads=1, concat=False, dropout=0.1, edge_dim=edge_dim)
        self.bn2 = nn.BatchNorm1d(embedding_dim)
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, x, edge_index, edge_attr):
        edge_index_dp, edge_mask = dropout_edge(edge_index, p=self.edge_dropout, force_undirected=False, training=self.training)
        edge_attr_dp = edge_attr[edge_mask] if edge_attr is not None else None
        
        x = F.dropout(x, p=0.25, training=self.training)
        x = self.conv1(x, edge_index_dp, edge_attr=edge_attr_dp)
        x = self.bn1(x)
        x = F.elu(x)
        
        edge_index_dp2, edge_mask2 = dropout_edge(edge_index, p=self.edge_dropout, force_undirected=False, training=self.training)
        edge_attr_dp2 = edge_attr[edge_mask2] if edge_attr is not None else None
        
        x = F.dropout(x, p=0.25, training=self.training)
        embedding = self.conv2(x, edge_index_dp2, edge_attr=edge_attr_dp2)
        embedding = self.bn2(embedding)
        embedding = F.elu(embedding) 
        
        out = self.classifier(embedding)
        return out, embedding

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs):
        scores = self.attention(lstm_outputs)
        weights = torch.softmax(scores, dim=1)
        context_vector = torch.sum(weights * lstm_outputs, dim=1)
        return context_vector, weights

class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super(SEBlock1D, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c) 
        y = self.fc(y).view(b, c, 1)    
        return x * y.expand_as(x)     

class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super(ResidualBlock1D, self).__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding)
        self.gn1 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding)
        self.gn2 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.dropout = nn.Dropout1d(0.2)
        
        self.se = SEBlock1D(out_channels)
        
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1),
                nn.GroupNorm(num_groups=8, num_channels=out_channels)
            )
            
    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv1(x)
        out = self.gn1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.conv2(out)
        out = self.gn2(out)
        
        
        out = self.se(out)
        
        out += residual  
        out = self.relu(out)
        return out


class CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes, time_steps, hidden_size=128):
        super(CNN_BiLSTM_Attention, self).__init__()
        
        self.res1 = ResidualBlock1D(num_features, 64)
        self.res2 = ResidualBlock1D(64, 128)
        
        self.pool = nn.MaxPool1d(kernel_size=2) 
        
        self.bilstm = nn.LSTM(input_size=128, hidden_size=hidden_size, 
                              batch_first=True, bidirectional=True)
        
        self.layer_norm = nn.LayerNorm(hidden_size * 2)
        
        self.attention = Attention(hidden_size * 2)
        self.dropout = nn.Dropout(0.5)
        
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        self.fc_ln = nn.LayerNorm(64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, num_classes)
        
    def forward(self, x):
        x = x.permute(0, 2, 1) 
        
        x = self.res1(x)
        x = self.res2(x)
        
        x = self.pool(x)
        
        x = x.permute(0, 2, 1)
        
        out, _ = self.bilstm(x)
        

        out = self.layer_norm(out)
        
        context_vector, attn_weights = self.attention(out)
        
        out = self.dropout(context_vector)
        out = self.fc1(out)
        out = self.fc_ln(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        
        return out


# tạo đồ thị GAT cho tập valid và test (cần tách riêng để mô phỏng thực tế (không gộp lại giống lúc train))
def build_inference_graph(df, window_size=50, max_dt=30.0):
    print("Pre-processing Timestamp cho Inference Graph...")
    if 'timestamp_num' not in df.columns:
        df['datetime_obj'] = pd.to_datetime(df['timestamp'], format='ISO8601', utc=True)
        df['timestamp_num'] = df['datetime_obj'].astype('int64') / 1e9
        
    df.sort_values(by='timestamp_num', inplace=True)
    df.reset_index(drop=True, inplace=True)
    
    cols_to_drop = [
        "timestamp", "timestamp_num", "datetime_obj",
        "network_ips_src", "network_ips_dst", "network_ports_src", "network_ports_dst",
        "network_ips_src_set", "network_ips_dst_set", "network_ports_src_set", "network_ports_dst_set",
        "label"
    ]
    cols_present = [c for c in cols_to_drop if c in df.columns]
    feature_cols = [c for c in df.columns if c not in cols_present]
    
    features_np = np.empty((len(df), len(feature_cols)), dtype=np.float32)
    for i, col in enumerate(feature_cols):
        features_np[:, i] = df[col].astype(np.float32).values
    features_np = np.ascontiguousarray(features_np)
    
    x_tensor = torch.from_numpy(features_np).contiguous()
    y_tensor = torch.tensor(df['label'].values, dtype=torch.long).contiguous()

    timestamps = df['timestamp_num'].values
    ips_src = df['network_ips_src_set'].values
    ips_dst = df['network_ips_dst_set'].values
    ports_src = df['network_ports_src_set'].values
    ports_dst = df['network_ports_dst_set'].values
    
    all_src, all_dst, all_edge_attrs = [], [], []
    from collections import deque
    window = deque()
    
    def jaccard_sim(set_a, set_b):
        if not set_a and not set_b: return 0.0
        intersect = len(set_a.intersection(set_b))
        union = len(set_a.union(set_b))
        return intersect / union if union > 0 else 0.0
        
    from tqdm.auto import tqdm
    for curr_idx in tqdm(range(len(df)), desc="Xây Dựng Đồ Thị (Culling Edge)"):
        curr_time = timestamps[curr_idx]
        curr_ip_src = ips_src[curr_idx]
        curr_ip_dst = ips_dst[curr_idx]
        
        while window and (curr_time - timestamps[window[0]]) > max_dt:
            window.popleft()
            
        recent_nodes = list(window)[-window_size:] 
        for past_idx in recent_nodes:
            past_ip_src = ips_src[past_idx]
            past_ip_dst = ips_dst[past_idx]
            
            curr_ip_all = curr_ip_src.union(curr_ip_dst)
            past_ip_all = past_ip_src.union(past_ip_dst)
            
            if len(curr_ip_all.intersection(past_ip_all)) > 0:
                dt_raw = abs(curr_time - timestamps[past_idx])
                dt = np.log1p(dt_raw * 1e6) / 18.0
                
                w_ip_src = jaccard_sim(curr_ip_src, past_ip_src)
                w_ip_dst = jaccard_sim(curr_ip_dst, past_ip_dst)
                
                curr_port_src = ports_src[curr_idx]
                past_port_src = ports_src[past_idx]
                w_port_src = jaccard_sim(curr_port_src, past_port_src)
                
                curr_port_dst = ports_dst[curr_idx]
                past_port_dst = ports_dst[past_idx]
                w_port_dst = jaccard_sim(curr_port_dst, past_port_dst)
                
                attr = [dt, w_ip_src, w_ip_dst, w_port_src, w_port_dst]
                
                all_src.append(past_idx)
                all_dst.append(curr_idx)
                all_edge_attrs.append(attr)
                
        window.append(curr_idx)
    
    edge_index = torch.tensor([all_src, all_dst], dtype=torch.long).contiguous()
    edge_attr = torch.tensor(all_edge_attrs, dtype=torch.float32).contiguous()
    
    graph = Data(x=x_tensor, edge_index=edge_index, edge_attr=edge_attr, y=y_tensor)
    graph.mask = torch.ones(len(df), dtype=torch.bool) 
    
    del features_np, all_src, all_dst, all_edge_attrs
    gc.collect()
    return graph, df

# trích xuất ma trận nối (raw features + embeddings) GAT thực tế bằng NeighborLoader
@torch.no_grad()
def extract_concat_features_mask(model, graph_data, device='cpu'):
    model.eval()
    loader = NeighborLoader(graph_data, num_neighbors=[50, 30], input_nodes=graph_data.mask, batch_size=512, shuffle=False, num_workers=0)
    all_combined = []
    from tqdm.auto import tqdm
    pbar = tqdm(loader, desc="Extracting Concat Features")
    for batch in pbar:
        batch = batch.to(device)
        _, emb = model(batch.x, batch.edge_index, batch.edge_attr)
        bs = batch.batch_size
        raw_features = batch.x[:bs].cpu().numpy()
        embeddings = emb[:bs].cpu().numpy()
        combined_matrix = np.concatenate([raw_features, embeddings], axis=1)
        all_combined.append(combined_matrix)
    pbar.close()
    return np.concatenate(all_combined, axis=0)

# trích xuất xác suất dự đoán từ mô hình CNN-BiLSTM-Attention
@torch.no_grad()
def extract_probs_cnn_bilstm(model, X_windows, batch_size=512, device='cpu'):
    model.eval()
    import numpy as np

    # Kiểm tra nhanh kích thước -> debug
    try:
        print("X_windows.shape:", X_windows.shape, "dtype:", X_windows.dtype, "bytes:", getattr(X_windows, "nbytes", None))
    except Exception:
        pass

    all_probs = []
    N = len(X_windows)
    for i in range(0, N, batch_size):
        batch_np = X_windows[i:i+batch_size]
        # đảm bảo float32 để giảm 2x bộ nhớ so với float64
        if batch_np.dtype != np.float32:
            batch_np = batch_np.astype(np.float32, copy=False)
        inputs = torch.from_numpy(batch_np).to(device)
        outputs = model(inputs)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        all_probs.append(probs)

    return np.vstack(all_probs)

In [14]:
import numpy as np
from tqdm.auto import tqdm
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score, roc_auc_score

# hàm tạo sliding windows với chỉ số mảng để align với output của GAT + XGBoost
def get_window_data_with_index(df, window_size=10):
    """
    trượt cửa sổ bằng chỉ số mảng (numpy array index) đẻ nối output của model (GAT_XGB)
    """
    ignore_cols = [
        "timestamp", "timestamp_num", "datetime_obj",
        "network_ips_src", "network_ips_dst", "network_ports_src", "network_ports_dst",
        "network_ips_src_set", "network_ips_dst_set", "network_ports_src_set", "network_ports_dst_set",
        "label"
    ]
    feature_cols = [c for c in df.columns if c not in ignore_cols]
    
    features = df[feature_cols].values.astype(np.float32)
    labels = df['label'].values
    
    X_windows = []
    y_targets = []
    target_indices = []
    
    print(f"Processing sliding windows (size={window_size})...")
    for i in tqdm(range(len(df) - window_size + 1)):
        window = features[i : i + window_size]
        target_idx = i + window_size - 1
        target_label = labels[target_idx]
        
        X_windows.append(window)
        y_targets.append(target_label)
        target_indices.append(target_idx)
        
    return np.array(X_windows), np.array(y_targets), np.array(target_indices)

# xây dựng đồ thị và trích xuất embeddings GAT cho tập valid và test
device = 'cuda' if torch.cuda.is_available() else 'cpu'

ignore_cols = [
    "timestamp", "timestamp_num", "datetime_obj",
    "network_ips_src", "network_ips_dst", "network_ports_src", "network_ports_dst",
    "network_ips_src_set", "network_ips_dst_set", "network_ports_src_set", "network_ports_dst_set",
    "label"
]
num_features = len([c for c in data_valid.columns if c not in ignore_cols])
num_classes = len(data_valid['label'].unique())

# load gat embedder
print(f"Đang Load trọng số GAT_Embedder (features = {num_features}, classes = {num_classes})...")
gat_model = GAT_Embedder(in_channels=num_features, hidden_channels=64, embedding_dim=32, num_classes=num_classes, heads=8, edge_dim=5).to(device)
gat_model.load_state_dict(torch.load(r"C:\Users\Admin\Downloads\IoT Dataset\CCIOT\model_final\gat_embedder_best.pth", map_location=device))

# xây dựng inference graph (tách riêng cho valid và test để mô phỏng thực tế, không gộp lại giống lúc train)
print("\n--- XÂY DỰNG INFERENCE GRAPH (Mất một chút RAM và Thời gian) ---")
graph_train, data_train = build_inference_graph(data_train) 
graph_valid, data_valid = build_inference_graph(data_valid)
graph_test, data_test = build_inference_graph(data_test)

# trích xuất đặc trưng nối (Concat: Raw Features + GAT Embeddings) từ train, valid và test
print("\n--- TRÍCH XUẤT RAW FEATURES & EMBEDDINGS TỪ TRAIN, VALID VÀ TEST ---")
train_gat_concat = extract_concat_features_mask(gat_model, graph_train, device=device)
valid_gat_concat = extract_concat_features_mask(gat_model, graph_valid, device=device)
test_gat_concat  = extract_concat_features_mask(gat_model, graph_test, device=device)

# tạo sliding windows cho CNN-BiLSTM (không có under-sampling, giữ nguyên thứ tự thời gian)
X_train_win, y_train_win, train_align_idx = get_window_data_with_index(data_train, window_size=10)
X_val_win, y_val_win, val_align_idx = get_window_data_with_index(data_valid, window_size=10)
X_test_win, y_test_win, test_align_idx = get_window_data_with_index(data_test, window_size=10)

print(f"\n[Valid] Windows Shape: {X_val_win.shape} | Align Indices: {val_align_idx.shape}")
print(f"[Test]  Windows Shape: {X_test_win.shape}  | Align Indices: {test_align_idx.shape}")

# sử dụng mô hình XGBoost đã huấn luyện để trích xuất xác suất dự đoán (probabilities) cho tập train và valid
print("\n--- Load XGBoost Model và Trích xuất xác suất (Probabilities) ---")
xgb_model = xgb.XGBClassifier()
xgb_model.load_model(r"model_final\GAT_XGB_Hybrid_Concat_Model_best.json")

prob_train_xgb = xgb_model.predict_proba(train_gat_concat)
prob_valid_xgb = xgb_model.predict_proba(valid_gat_concat)
prob_test_xgb  = xgb_model.predict_proba(test_gat_concat)

print("\n--- Load CNN-BiLSTM_Attention Model và Trích xuất xác suất ---")
cnn_model = CNN_BiLSTM_Attention(num_features=num_features, num_classes=num_classes, time_steps=10, hidden_size=128).to(device)
cnn_model.load_state_dict(torch.load(r"model_final\best_cnn_bilstm_best.pth", map_location=device, weights_only=True))

# extract probabilities từ CNN-BiLSTM-Attention cho tập valid và test
prob_train_bilstm = extract_probs_cnn_bilstm(cnn_model, X_train_win, batch_size=512, device=device)
prob_valid_bilstm = extract_probs_cnn_bilstm(cnn_model, X_val_win, batch_size=512, device=device)
prob_test_bilstm  = extract_probs_cnn_bilstm(cnn_model, X_test_win, batch_size=512, device=device)


# rút ra các dự đoán XGBoost (Node level) tương ứng với phần tử cuối cùng của (Sequence Level window)
aligned_prob_train_xgb = prob_train_xgb[train_align_idx]
aligned_prob_valid_xgb = prob_valid_xgb[val_align_idx]
aligned_prob_test_xgb  = prob_test_xgb[test_align_idx]

# nối đặc trưng meta: XGB Probabilities + CNN-BiLSTM Probabilities

X_meta_train = np.concatenate([aligned_prob_train_xgb, prob_train_bilstm], axis=1)
y_meta_train = y_train_win

X_meta_valid = np.concatenate([aligned_prob_valid_xgb, prob_valid_bilstm], axis=1)
y_meta_valid = y_val_win 

X_meta_test = np.concatenate([aligned_prob_test_xgb, prob_test_bilstm], axis=1)
y_meta_test = y_test_win

print(f"\n[Meta-Dataset] Train Shape:      {X_meta_train.shape}")
print(f"\n[Meta-Dataset] Validation Shape: {X_meta_valid.shape}")
print(f"[Meta-Dataset] Test Shape:       {X_meta_test.shape}")


# meta-learner dùng XGBoost
print("\n--- BẮT ĐẦU HUẤN LUYỆN META-LEARNER DÙNG XGBoost ---")

meta_model = xgb.XGBClassifier(
    n_estimators=300,          
    max_depth=3,           
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    random_state=42,
    early_stopping_rounds=20  
)

meta_model.fit(
    X_meta_valid, y_meta_valid,
    eval_set=[(X_meta_valid, y_meta_valid)],
    verbose=False
)

final_preds = meta_model.predict(X_meta_test)
final_probas = meta_model.predict_proba(X_meta_test)
print("\n--- KẾT QUẢ CUỐI CÙNG STACKING CỦA GRAPH + SEQUENCE TRÊN TẬP TEST ---")
print(f"Accuracy:         {accuracy_score(y_meta_test, final_preds)*100:.2f}%")
print(f"F1-Score (Macro): {f1_score(y_meta_test, final_preds, average='macro')*100:.2f}%")
auc_roc = roc_auc_score(y_meta_test, final_probas, multi_class='ovr', average='macro')
print(f"AUC-ROC (Macro, OVR): {auc_roc:.4f}")
print("\nClassification Report:\n")
print(classification_report(y_meta_test, final_preds, digits=4))

Đang Load trọng số GAT_Embedder (features = 133, classes = 8)...

--- XÂY DỰNG INFERENCE GRAPH (Mất một chút RAM và Thời gian) ---
Pre-processing Timestamp cho Inference Graph...


C:\Users\Admin\AppData\Local\Temp\ipykernel_12840\973035529.py:54: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  gat_model.load_state_dict(torch.load(r"C:\Users\Admin\Downlo

Xây Dựng Đồ Thị (Culling Edge):   0%|          | 0/159031 [00:00<?, ?it/s]

Pre-processing Timestamp cho Inference Graph...


Xây Dựng Đồ Thị (Culling Edge):   0%|          | 0/34077 [00:00<?, ?it/s]

...

Processing sliding windows (size=10)...


  0%|          | 0/34068 [00:00<?, ?it/s]

Processing sliding windows (size=10)...


  0%|          | 0/34074 [00:00<?, ?it/s]


[Valid] Windows Shape: (34068, 10, 133) | Align Indices: (34068,)
[Test]  Windows Shape: (34074, 10, 133)  | Align Indices: (34074,)

--- Load XGBoost Model và Trích xuất xác suất (Probabilities) ---

--- Load CNN-BiLSTM_Attention Model và Trích xuất xác suất ---
X_windows.shape: (159022, 10, 133) dtype: float32 bytes: 845997040
X_windows.shape: (34068, 10, 133) dtype: float32 bytes: 181241760
X_windows.shape: (34074, 10, 133) dtype: float32 bytes: 181273680

[Meta-Dataset] Train Shape:      (159022, 16)

[Meta-Dataset] Validation Shape: (34068, 16)
[Meta-Dataset] Test Shape:       (34074, 16)

--- BẮT ĐẦU HUẤN LUYỆN META-LEARNER DÙNG XGBoost ---

--- KẾT QUẢ CUỐI CÙNG STACKING CỦA GRAPH + SEQUENCE TRÊN TẬP TEST ---
Accuracy:         90.07%
F1-Score (Macro): 74.84%
AUC-ROC (Macro, OVR): 0.9843

Classification Report:

              precision    recall  f1-score   support

           0     0.9821    0.9987    0.9903     20520
           1     0.4886    0.6868    0.5710       281
      

Meta Learner bằng Logistic Regression

In [19]:
# huấn luyện meta-learner dùng logistic regression để so sánh với XGBoost meta-learner
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score, roc_auc_score

print("\n--- BẮT ĐẦU HUẤN LUYỆN META-LEARNER (LOGISTIC REGRESSION) ---")

best_lr_model = LogisticRegression(
    C=1.0, 
    solver='lbfgs', 
    class_weight='balanced', 
    max_iter=2000, 
    random_state=42
)

print("Đang huấn luyện mô hình Meta-learner trên tập Valid...")
best_lr_model.fit(X_meta_valid, y_meta_valid)

lr_final_preds = best_lr_model.predict(X_meta_test)
lr_final_probas = best_lr_model.predict_proba(X_meta_test)

print("\n--- KẾT QUẢ CUỐI CÙNG STACKING CỦA GRAPH + SEQUENCE ---")
print(f"Accuracy:         {accuracy_score(y_meta_test, lr_final_preds)*100:.2f}%")
print(f"F1-Score (Macro): {f1_score(y_meta_test, lr_final_preds, average='macro')*100:.2f}%")
auc_roc = roc_auc_score(y_meta_test, lr_final_probas, multi_class='ovr', average='macro')
print(f"AUC-ROC (Macro, OVR): {auc_roc:.4f}")
print("\nClassification Report:\n")
print(classification_report(y_meta_test, lr_final_preds, digits=4))


--- BẮT ĐẦU HUẤN LUYỆN META-LEARNER (LOGISTIC REGRESSION) ---
Đang huấn luyện mô hình Meta-learner trên tập Valid...

--- KẾT QUẢ CUỐI CÙNG STACKING CỦA GRAPH + SEQUENCE ---
Accuracy:         90.49%
F1-Score (Macro): 75.02%
AUC-ROC (Macro, OVR): 0.9430

Classification Report:

              precision    recall  f1-score   support

           0     0.9908    0.9947    0.9927     20520
           1     0.2407    0.9217    0.3817       281
           2     0.7596    0.4795    0.5879      2709
           3     0.6617    0.6967    0.6788      2763
           4     0.9379    1.0000    0.9679      1132
           5     0.9707    0.3281    0.4904      1210
           6     0.8693    0.9942    0.9276      5039
           7     1.0000    0.9500    0.9744       420

    accuracy                         0.9049     34074
   macro avg     0.8038    0.7956    0.7502     34074
weighted avg     0.9192    0.9049    0.9015     34074

